# 🏆 FIFA World Cup Prediction — Data & Feature Pipeline

**Single notebook. Run top to bottom.**

| Section | What happens |
|---|---|
| 1. Setup | Install libraries, imports |
| 2. Scrape | Fetch all raw data from GitHub |
| 3. Clean | Matches, goals, squads, players |
| 4. Team features | Elo, form, squad profile, history |
| 5. Player features | Goals, age, experience, position |
| 6. Analysis | Correlations, distributions |
| 7. Save | Only the two files the modelling notebook needs |

**Outputs (saved at the end):**
- `team_features.csv` — one row per team per tournament, all features + stage_rank target
- `player_features.csv` — one row per player per tournament, all features + award flags


## 1. Setup

In [ ]:
!pip install requests pandas numpy matplotlib seaborn scikit-learn --quiet

import requests, io, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")
print("✅ Ready")

## 2. Scrape Raw Data

All data from `github.com/jfjelstul/worldcup` — covers all men's World Cups 1930–2022.


In [ ]:
BASE = "https://raw.githubusercontent.com/jfjelstul/worldcup/master/data-csv/"

def fetch(filename):
    r = requests.get(BASE + filename, timeout=15)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), low_memory=False)
    print(f"  ✅ {filename:<35} {df.shape[0]:>5,} rows × {df.shape[1]} cols")
    return df

print("Fetching...")
matches          = fetch("matches.csv")
goals            = fetch("goals.csv")
squads           = fetch("squads.csv")
players          = fetch("players.csv")
tournaments      = fetch("tournaments.csv")
team_app         = fetch("team_appearances.csv")
player_app       = fetch("player_appearances.csv")
award_winners    = fetch("award_winners.csv")
print("\n✅ All files loaded")

## 3. Reference Maps & Filters

In [ ]:
# Men's tournaments only
men_tourns = tournaments[
    tournaments["tournament_name"].str.contains("Men", na=False)
].copy()

YEAR_MAP   = men_tourns.set_index("tournament_id")["year"].to_dict()
WINNER_MAP = men_tourns.set_index("tournament_id")["winner"].to_dict()
HOST_MAP   = men_tourns.set_index("year")["host_country"].to_dict()

STAGE_RANK = {
    "group stage": 1,        "second group stage": 2,
    "final round": 2,        "round of 16": 3,
    "quarter-final": 4,      "quarter-finals": 4,
    "semi-final": 5,         "semi-finals": 5,
    "third-place match": 6,  "final": 7
}

AWARD_IDS = {
    "A-1": "won_best_player",
    "A-4": "won_golden_boot",
    "A-8": "won_young_player"
}

def men_only(df):
    df = df.copy()
    df["year"] = df["tournament_id"].map(YEAR_MAP)
    return df[df["year"].notna()].reset_index(drop=True)

# Filter all raw tables to men's WC once
m_matches  = men_only(matches)
m_goals    = men_only(goals)
m_squads   = men_only(squads)
m_team_app = men_only(team_app)
m_player_app = men_only(player_app)
m_awards   = men_only(award_winners)

for name, df in [("matches",m_matches),("goals",m_goals),("squads",m_squads),
                  ("team_app",m_team_app),("player_app",m_player_app),("awards",m_awards)]:
    print(f"  {name:<15} {len(df):>5,} rows  |  years: {sorted(df['year'].dropna().astype(int).unique())[-3:]}")

## 4. Build Features
### 4a. Clean Matches

In [ ]:
def clean_matches(df):
    df = df[[
        "year","tournament_id","match_id","stage_name","match_date",
        "home_team_name","away_team_name",
        "home_team_score","away_team_score",
        "home_team_win","away_team_win","draw",
        "extra_time","penalty_shootout"
    ]].copy()

    df.rename(columns={
        "home_team_name":"home_team", "away_team_name":"away_team",
        "home_team_score":"home_goals","away_team_score":"away_goals"
    }, inplace=True)

    df["year"]       = df["year"].astype(int)
    df["home_goals"] = pd.to_numeric(df["home_goals"], errors="coerce")
    df["away_goals"] = pd.to_numeric(df["away_goals"], errors="coerce")
    df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")
    df["total_goals"]= df["home_goals"] + df["away_goals"]
    df.dropna(subset=["home_goals","away_goals"], inplace=True)
    return df.sort_values(["year","match_date"]).reset_index(drop=True)

clean_m = clean_matches(m_matches)
print(f"✅ Matches: {len(clean_m):,} rows | {clean_m['year'].nunique()} tournaments")
display(clean_m.tail(3))

### 4b. Elo Ratings

Computed from all WC match history. Snapshotted **before** each tournament starts to prevent data leakage.

In [ ]:
def compute_elo(df, k=32, initial=1000):
    elo = {}

    def get(team):  return elo.get(team, initial)
    def exp(a, b):  return 1 / (1 + 10 ** ((b - a) / 400))

    history = {}
    for year, grp in df.sort_values(["year","match_date"]).groupby("year"):
        # Snapshot before tournament
        for team in set(grp["home_team"]) | set(grp["away_team"]):
            history[(int(year), team)] = round(get(team), 1)

        # Update through tournament
        for _, r in grp.iterrows():
            h, a = r["home_team"], r["away_team"]
            hg, ag = r["home_goals"], r["away_goals"]
            if pd.isna(hg) or pd.isna(ag): continue

            e_h = exp(get(h), get(a))
            act_h = 1 if hg > ag else (0 if hg < ag else 0.5)
            mov = np.log(abs(hg - ag) + 1) + 1 if hg != ag else 1

            elo[h] = get(h) + k * mov * (act_h - e_h)
            elo[a] = get(a) + k * mov * ((1 - act_h) - (1 - e_h))

    return pd.DataFrame([
        {"year": yr, "team_name": team, "elo": elo_val}
        for (yr, team), elo_val in history.items()
    ])

elo_df = compute_elo(clean_m)
print(f"✅ Elo: {len(elo_df):,} (year, team) pairs")

# Sanity check
winners_elo = elo_df.copy()
winners_elo["winner"] = winners_elo.apply(
    lambda r: WINNER_MAP.get(f"WC-{int(r['year'])}"), axis=1
)
winners_elo["is_winner"] = winners_elo["team_name"] == winners_elo["winner"]
winners_elo["elo_rank"] = winners_elo.groupby("year")["elo"].rank(ascending=False, method="min")
wr = winners_elo[winners_elo["is_winner"]][["year","team_name","elo","elo_rank"]]
print(f"\nAvg Elo rank of tournament winners: {wr['elo_rank'].mean():.1f}")
print(f"% of winners in top-3 Elo        : {(wr['elo_rank'] <= 3).mean()*100:.0f}%")

### 4c. Pre-Tournament Form

Each team's performance in their **previous** World Cup.

In [ ]:
def compute_form(df):
    df = df.copy()
    df["year"] = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)

    agg = df.groupby(["year","team_name"]).agg(
        mp   = ("match_id",      "count"),
        wins = ("win",           "sum"),
        gf   = ("goals_for",     "sum"),
        ga   = ("goals_against", "sum"),
        max_stage = ("stage_rank", "max"),
    ).reset_index()

    agg["win_rate"]    = agg["wins"] / agg["mp"]
    agg["gd_per_game"] = (agg["gf"] - agg["ga"]) / agg["mp"]
    agg["gf_per_game"] = agg["gf"] / agg["mp"]

    agg = agg.sort_values(["team_name","year"])
    for col, new in [("win_rate","form_win_rate"),("gd_per_game","form_gd_per_game"),
                      ("gf_per_game","form_gf_per_game"),("max_stage","prev_stage_rank")]:
        agg[new] = agg.groupby("team_name")[col].shift(1)

    for col in ["form_win_rate","form_gd_per_game","form_gf_per_game","prev_stage_rank"]:
        agg[col] = agg[col].fillna(0)

    return agg[["year","team_name","form_win_rate","form_gd_per_game",
                 "form_gf_per_game","prev_stage_rank"]]

form_df = compute_form(m_team_app)
print(f"✅ Form features: {len(form_df):,} rows")

### 4d. Squad Profile

In [ ]:
def compute_squad_features(squads_df, players_df, tournaments_df):
    sq = squads_df.copy()
    sq["year"] = sq["year"].astype(int)

    # Bio data
    bio = players_df[["player_id","birth_date","count_tournaments","list_tournaments"]].copy()
    bio["birth_date"] = pd.to_datetime(bio["birth_date"], errors="coerce")
    sq = sq.merge(bio, on="player_id", how="left")

    # Tournament start dates for age calculation
    t_starts = (
        tournaments_df[tournaments_df["tournament_name"].str.contains("Men", na=False)]
        .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
        .set_index("year")["start_date"].to_dict()
    )
    sq["t_start"] = sq["year"].map(t_starts)
    sq["age"] = ((sq["t_start"] - sq["birth_date"]).dt.days / 365.25)

    # WC experience: previous WCs before this one
    def prev_wc(row):
        try:
            yrs = [int(y.strip()) for y in str(row["list_tournaments"]).split(",")]
            return sum(1 for y in yrs if y < int(row["year"]))
        except:
            return 0
    sq["wc_exp"] = sq.apply(prev_wc, axis=1)

    agg = sq.groupby(["year","team_name"]).agg(
        squad_size    = ("player_id",  "count"),
        squad_avg_age = ("age",        "mean"),
        squad_min_age = ("age",        "min"),
        squad_avg_exp = ("wc_exp",     "mean"),
    ).reset_index()

    # Positional counts
    for pos, col in [("goalkeeper","n_gk"),("defender","n_def"),
                      ("midfielder","n_mid"),("forward","n_fwd")]:
        cnt = (sq[sq["position_name"].str.lower().str.contains(pos, na=False)]
               .groupby(["year","team_name"]).size().reset_index(name=col))
        agg = agg.merge(cnt, on=["year","team_name"], how="left")
        agg[col] = agg[col].fillna(0).astype(int)

    agg["squad_avg_age"] = agg["squad_avg_age"].round(1)
    agg["squad_avg_exp"] = agg["squad_avg_exp"].round(2)
    return agg

squad_df = compute_squad_features(m_squads, players, tournaments)
print(f"✅ Squad features: {len(squad_df):,} rows")

### 4e. Tournament History

In [ ]:
def compute_history(team_app_df):
    df = team_app_df.copy()
    df["year"]       = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)

    max_stage = (df.groupby(["year","team_name"])["stage_rank"]
                 .max().reset_index()
                 .sort_values(["team_name","year"]))

    records = []
    for team, grp in max_stage.groupby("team_name"):
        grp = grp.sort_values("year").reset_index(drop=True)
        for i, row in grp.iterrows():
            past = grp[grp["year"] < row["year"]]
            records.append({
                "year":           row["year"],
                "team_name":      team,
                "wc_appearances": len(past),
                "best_stage_ever":past["stage_rank"].max() if len(past) > 0 else 0,
            })

    hist = pd.DataFrame(records)
    hist["is_host"]     = hist.apply(lambda r: int(HOST_MAP.get(int(r["year"])) == r["team_name"]), axis=1)
    hist["won_last_wc"] = hist.apply(
        lambda r: int(WINNER_MAP.get(f"WC-{int(r['year'])-4}","") == r["team_name"]), axis=1
    )
    hist["best_stage_ever"] = hist["best_stage_ever"].fillna(0)
    return hist

history_df = compute_history(m_team_app)
print(f"✅ History features: {len(history_df):,} rows")

### 4f. Merge Everything → `team_features`

In [ ]:
def compute_targets(team_app_df):
    df = team_app_df.copy()
    df["year"]       = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)
    targets = df.groupby(["year","team_name"])["stage_rank"].max().reset_index()
    targets["won_tournament"] = targets.apply(
        lambda r: int(WINNER_MAP.get(f"WC-{r['year']}","") == r["team_name"]), axis=1
    )
    return targets

targets = compute_targets(m_team_app)

team_features = (
    targets
    .merge(elo_df,     on=["year","team_name"], how="left")
    .merge(form_df,    on=["year","team_name"], how="left")
    .merge(squad_df,   on=["year","team_name"], how="left")
    .merge(history_df, on=["year","team_name"], how="left")
)

fill_zero = ["elo","form_win_rate","form_gd_per_game","form_gf_per_game",
             "prev_stage_rank","squad_avg_exp","wc_appearances",
             "best_stage_ever","is_host","won_last_wc"]
for col in fill_zero:
    if col in team_features.columns:
        team_features[col] = team_features[col].fillna(0)

if "squad_avg_age" in team_features.columns:
    team_features["squad_avg_age"] = team_features["squad_avg_age"].fillna(
        team_features["squad_avg_age"].median()
    )

team_features = team_features.sort_values(["year","team_name"]).reset_index(drop=True)
print(f"✅ team_features: {team_features.shape[0]:,} rows × {team_features.shape[1]} cols")
print(f"\nNull check:")
nulls = team_features.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else "  No nulls ✅")
display(team_features[team_features["year"]==2022].sort_values("elo", ascending=False).head(8))

## 5. Player Features → `player_features`

In [ ]:
# Goals per player per tournament
goal_counts = (
    m_goals[~m_goals["own_goal"].astype(bool)]
    .assign(year=lambda d: d["year"].astype(int))
    .groupby(["year","player_id","team_name"]).agg(
        goals_scored   = ("goal_id", "count"),
        penalty_goals  = ("penalty", "sum"),
    ).reset_index()
)
goal_counts["open_play_goals"] = goal_counts["goals_scored"] - goal_counts["penalty_goals"]

# Appearances per player per tournament
app_counts = (
    m_player_app
    .assign(year=lambda d: d["year"].astype(int))
    .groupby(["year","player_id","team_name"]).agg(
        matches_played  = ("match_id", "count"),
        matches_started = ("starter",  "sum"),
        position_name   = ("position_name", "first"),
    ).reset_index()
)

# Bio data: name, age, experience
bio = players[["player_id","family_name","given_name","birth_date",
               "count_tournaments","list_tournaments"]].copy()
bio["player_name"] = bio["given_name"].str.strip() + " " + bio["family_name"].str.strip()
bio["birth_date"]  = pd.to_datetime(bio["birth_date"], errors="coerce")

t_starts = (
    tournaments[tournaments["tournament_name"].str.contains("Men", na=False)]
    .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
    .set_index("year")["start_date"].to_dict()
)

# Merge
player_features = (
    app_counts
    .merge(goal_counts, on=["year","player_id","team_name"], how="left")
    .merge(bio, on="player_id", how="left")
)

player_features["goals_scored"]    = player_features["goals_scored"].fillna(0).astype(int)
player_features["penalty_goals"]   = player_features["penalty_goals"].fillna(0).astype(int)
player_features["open_play_goals"] = player_features["open_play_goals"].fillna(0).astype(int)

# Age at tournament
player_features["t_start"] = player_features["year"].map(t_starts)
player_features["age"] = (
    (player_features["t_start"] - player_features["birth_date"]).dt.days / 365.25
).round(1)

# WC experience
def prev_wc(row):
    try:
        yrs = [int(y.strip()) for y in str(row["list_tournaments"]).split(",")]
        return sum(1 for y in yrs if y < int(row["year"]))
    except:
        return 0
player_features["wc_exp"] = player_features.apply(prev_wc, axis=1)

# Award flags
for col in ["won_golden_boot","won_best_player","won_young_player"]:
    player_features[col] = False

for _, row in m_awards.iterrows():
    col = AWARD_IDS.get(row["award_id"])
    if not col: continue
    mask = (
        (player_features["year"] == int(row["year"])) &
        (player_features["player_id"] == row["player_id"])
    )
    player_features.loc[mask, col] = True

player_features = player_features.drop(columns=["t_start","birth_date",
    "list_tournaments","count_tournaments"], errors="ignore")

print(f"✅ player_features: {player_features.shape[0]:,} rows × {player_features.shape[1]} cols")
print(f"   Golden Boot flags : {player_features['won_golden_boot'].sum()}")
print(f"   Best Player flags : {player_features['won_best_player'].sum()}")
print(f"   Young Player flags: {player_features['won_young_player'].sum()}")

print("\nTop scorers 2022:")
display(player_features[player_features["year"]==2022]
        .nlargest(5,"goals_scored")[
    ["player_name","team_name","goals_scored","penalty_goals","matches_played","age"]])

## 6. Feature Analysis

In [ ]:
FEAT_COLS = [c for c in [
    "elo","form_win_rate","form_gd_per_game","form_gf_per_game",
    "prev_stage_rank","squad_avg_age","squad_avg_exp",
    "wc_appearances","best_stage_ever","is_host","won_last_wc"
] if c in team_features.columns]

corr = (team_features[FEAT_COLS + ["stage_rank"]]
        .corr()["stage_rank"]
        .drop("stage_rank")
        .sort_values(ascending=False))

print("Correlation with stage_rank:")
print(corr.round(3).to_string())
print("\n→ |corr| > 0.15 = worth keeping")
print("→ |corr| < 0.05 = likely noise")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Feature Distributions — Winners vs Others", fontsize=14)

plot_feats = [
    ("elo",              "Elo Rating"),
    ("form_win_rate",    "Prev WC Win Rate"),
    ("form_gd_per_game", "Prev WC GD/Game"),
    ("squad_avg_age",    "Squad Avg Age"),
    ("squad_avg_exp",    "Squad WC Experience"),
    ("wc_appearances",   "WC Appearances"),
]

for ax, (col, title) in zip(axes.flat, plot_feats):
    if col not in team_features.columns:
        ax.set_visible(False); continue
    for won, grp in team_features.groupby("won_tournament"):
        ax.hist(grp[col].dropna(), bins=20, alpha=0.7, edgecolor="white",
                color="#FFD700" if won else "#4A90D9",
                label="Winner" if won else "Other", density=True)
    ax.set_title(title); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap — check for multicollinearity
fig, ax = plt.subplots(figsize=(12, 9))
cm = team_features[FEAT_COLS].corr()
mask = np.triu(np.ones_like(cm, dtype=bool))
sns.heatmap(cm, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size":8})
ax.set_title("Feature Correlation Matrix\n(pairs > 0.7 = multicollinearity risk)", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Save Outputs

Only the two files the modelling notebook needs.
Everything else was computed in memory and doesn't need to be saved.


In [ ]:
from google.colab import files as colab_files

team_features.to_csv("team_features.csv", index=False)
player_features.to_csv("player_features.csv", index=False)

colab_files.download("team_features.csv")
colab_files.download("player_features.csv")

print(f"✅ team_features.csv   — {team_features.shape[0]:,} rows × {team_features.shape[1]} cols")
print(f"✅ player_features.csv — {player_features.shape[0]:,} rows × {player_features.shape[1]} cols")

## ✅ Done

### What you have

| File | Rows | Purpose |
|---|---|---|
| `team_features.csv` | ~500 | Team-level model input + `stage_rank` target |
| `player_features.csv` | ~11,000 | Player-level model input + award flag targets |

### Next notebook — Modelling
1. Train/test split **by year** (not random — random split leaks future data into training)
2. Poisson regression for match score prediction
3. Logistic regression for tournament winner probability
4. Monte Carlo simulation — simulate tournament 10,000 times, record outcome distribution
5. Baseline evaluation — does the model beat simply picking the highest-Elo team?
6. Award prediction models — ranking model for Golden Boot, Best Player, Young Player
